# Clase 1 · De Python a tu primer LLM

Le hablás a Gemini desde Python desde el primer ejercicio. Así repasás el Python del curso
y, de paso, el cuaderno arma tu **diagnóstico** (sin nota) con lo que te salió y lo que no.

**Antes de empezar:** tu clave de Gemini en los **Secretos** de Colab (la llave de la izquierda),
con el nombre `GOOGLE_API_KEY` y **Acceso del notebook** activado. Paso a paso, en la slide
«Antes del cuaderno» y en [AI Studio](https://aistudio.google.com/apikey). La clave nunca va en una celda.

**Cómo se usa:** corré las celdas en orden con **Shift + Enter**. Donde dice `None  # ← tu respuesta`,
escribí tu código. Cada ejercicio dice ✅ o ❌ con una pista.

<details><summary><b>¿Sale un error?</b></summary>

- **`None` o `NoneType`** en rojo: falta tu respuesta en esa celda o en una anterior de la que depende.
- **`NameError`** (por ejemplo `name 'modelo' is not defined`): salteaste una celda o Colab se reinició.
  Menú *Entorno de ejecución* → *Run before*.
- **«No pude leer el secreto GOOGLE_API_KEY»:** revisá el nombre (exacto, en mayúsculas) y el acceso del notebook.
- **`API key not valid` o `PERMISSION_DENIED`:** la clave está mal copiada. Volvé a pegarla en Secretos.
- **`503 UNAVAILABLE`** («high demand»): Gemini está saturado. Esperá un minuto y volvé a correr la celda.
- **Te trabaste más de 5 minutos:** mirá la pista (el último texto de `revisar(...)`) y pasá al siguiente.
  Anotá cuál fue: es justo lo que buscamos saber.

</details>

In [ ]:
# Preparar (una vez, unos 30 segundos): instala LangChain y la conexión con Gemini, carga la clave y el modelo.
%pip install -q langchain==1.4.6 langchain-google-genai==4.4.2
import os

try:
    from google.colab import userdata                      # en Colab: la clave sale de los Secretos
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass                                                   # fuera de Colab: GOOGLE_API_KEY ya tiene que estar
except Exception as error:                                 # no hay secreto con ese nombre o le falta el acceso
    print("No pude leer el secreto GOOGLE_API_KEY:", type(error).__name__)
    print("Revisá el nombre (exacto, en mayúsculas) y que «Acceso del notebook» esté activado.")

from langchain.chat_models import init_chat_model
modelo = init_chat_model("google_genai:gemini-3.5-flash-lite")

resultados = {}  # lo que te salió en cada ejercicio: con esto se arma tu diagnóstico al final


def revisar(nombre, valor, esperado, pista=""):
    resultados[nombre] = valor == esperado
    if valor == esperado:
        print(f"✅ {nombre}: bien")
    else:
        print(f"❌ {nombre}: obtuviste {valor!r}, se esperaba {esperado!r}")
        if pista:
            print("   Pista:", pista)


print("Listo: clave cargada." if os.environ.get("GOOGLE_API_KEY") else "Falta la clave: mirá «Antes de empezar».")

## Parte 1 · Python, hablando con Gemini (10 min)

### Ejercicio 1 · Variables y texto

Una **variable** guarda un valor: `tema = "un token"`. Con una **f-string** metés variables
adentro de un texto: `f"Hablemos de {tema}"`.

**Tu turno:** armá en `pregunta` el texto `Explicá en una oración qué es un token.` usando la variable `tema`.
La celda se la manda a Gemini y te muestra lo que contesta (una llamada).

In [ ]:
tema = "un token"   # cuando te salga, probá con otro tema
pregunta = None  # ← tu respuesta
print(pregunta)
revisar("Ejercicio 1", pregunta, f"Explicá en una oración qué es {tema}.",
        'pregunta = f"Explicá en una oración qué es {tema}."')

if pregunta:
    r = modelo.invoke(pregunta)
    print(r.text)

### Ejercicio 2 · Listas: la respuesta, palabra por palabra

Una **lista** guarda valores en orden. `.split()` parte un texto en una lista, cortando en los espacios,
y `len(lista)` cuenta cuántos elementos tiene.

**Tu turno:** guardá en `palabras` la respuesta de Gemini (`r.text`) partida en palabras, y en `cantidad` cuántas son.

In [ ]:
palabras = None  # ← tu respuesta
cantidad = None  # ← tu respuesta
print(cantidad, "palabras:", palabras)
revisar("Ejercicio 2", (palabras, cantidad), (r.text.split(), len(r.text.split())),
        "palabras = r.text.split() y cantidad = len(palabras)")

### Ejercicio 3 · Recorrer con `for` e `if`

`for texto in respuestas:` repite el bloque una vez por cada respuesta. `if len(texto.split()) > 20:` hace
algo solo si esa respuesta tiene más de 20 palabras. La celda le pregunta a Gemini por tres temas (tres llamadas).

**Tu turno:** contá en `largas` cuántas respuestas tienen **más de 20 palabras**.

In [ ]:
temas = ["la temperatura", "un embedding", "una alucinación"]
respuestas = [modelo.invoke(f"Explicá en una oración qué es {un_tema}.").text for un_tema in temas]

largas = 0
for texto in respuestas:
    pass  # ← reemplazá esta línea: si texto tiene más de 20 palabras, sumá 1 a largas
print(largas, "de", len(respuestas), "respuestas tienen más de 20 palabras")
revisar("Ejercicio 3", largas, sum(len(x.split()) > 20 for x in respuestas),
        "Dentro del for: if len(texto.split()) > 20: largas = largas + 1")

### Ejercicio 4 · Diccionarios: cuántos tokens usó

Un **diccionario** guarda pares *clave → valor*: `{"input_tokens": 15, "output_tokens": 2}`. Lo leés con
`uso["input_tokens"]`. Gemini devuelve uno así en `r.usage_metadata`: los tokens de tu pregunta (*input*) y los
de su respuesta (*output*). Se paga por token.

**Tu turno:** guardá en `tokens_entrada` y `tokens_salida` los dos números.

In [ ]:
uso = r.usage_metadata
print(uso)
tokens_entrada = None  # ← tu respuesta
tokens_salida = None   # ← tu respuesta
revisar("Ejercicio 4", (tokens_entrada, tokens_salida), (uso["input_tokens"], uso["output_tokens"]),
        'uso["input_tokens"] y uso["output_tokens"]')

### Ejercicio 5 · Funciones: tu propio `preguntar`

Una **función** es un bloque con nombre que recibe datos y devuelve un resultado (`def` y `return`). Así no
repetís la llamada cada vez.

**Tu turno:** completá `preguntar` para que le mande `texto` a Gemini y devuelva **solo** lo que escribió (`.text`).

**Qué deberías ver** (medido el 21/10): `Montevideo`.

In [ ]:
def preguntar(texto):
    return None  # ← tu respuesta

capital = preguntar("¿Cuál es la capital de Uruguay? Respondé en una palabra.")
print(capital)
revisar("Ejercicio 5", isinstance(capital, str) and len(capital) > 0, True, "return modelo.invoke(texto).text")

## Parte 2 · Un modelo de lenguaje de juguete

Ahora, sin Gemini, un modelo de juguete. Contamos **qué palabra viene después de cada
palabra**. Con eso podemos predecir: después de «el», ¿qué es lo más probable?

Esto es un **clasificador del próximo token**, como en ML: la entrada es la
palabra actual y las clases son las palabras posibles. Un LLM hace lo mismo,
pero mira todo el texto anterior (no solo una palabra) y tiene entre 100 mil
y 260 mil clases, según el modelo: los tokens de su vocabulario.

### Ejercicio 6 · Tabla de «qué sigue»

La celda arma `siguientes`: para cada palabra, un diccionario con cuántas veces
apareció cada palabra **después**. Ya está escrita: leela línea por línea y
corré la celda.

Dos piezas nuevas: `tokens[1:]` es la misma lista sin la primera palabra, y
`zip(a, b)` las junta de a pares: («el», «gato»), («gato», «come»),
(«come», «pescado»)… Así cada palabra queda al lado de la que la sigue.

**Tu turno:** Después, guardá en `despues_de_come` el diccionario de lo que sigue a «come».

In [ ]:
texto = """el gato come pescado . el perro come carne . el gato duerme .
el perro come pescado . el gato come carne . el perro duerme ."""
tokens = texto.split()

siguientes = {}
for actual, proxima in zip(tokens, tokens[1:]):      # pares (palabra, la que sigue)
    siguientes.setdefault(actual, {})                # si no estaba, empieza vacío
    siguientes[actual][proxima] = siguientes[actual].get(proxima, 0) + 1

despues_de_come = None  # ← tu respuesta
print(despues_de_come)
revisar("Ejercicio 6", despues_de_come, {"pescado": 2, "carne": 2}, 'siguientes["come"]')

### Ejercicio 7 · Softmax con temperatura

Un LLM no cuenta: da un **puntaje** a cada token posible y los convierte en
probabilidades con **softmax**. La **temperatura** `T` divide los puntajes
antes del softmax:

- `T` chica (0.5): la más probable se lleva casi todo → respuestas repetibles.
- `T = 1`: tal cual.
- `T` grande (2): todo se empareja → más variedad, más rarezas.

La fórmula: `p_i = exp(puntaje_i / T) / suma de exp(puntaje_j / T)`.

**Tu turno:** Completá `softmax`. La celda imprime las probabilidades con 3 temperaturas.

In [ ]:
import math

def softmax(puntajes, T=1.0):
    exps = {tok: math.exp(p / T) for tok, p in puntajes.items()}
    total = None  # ← tu respuesta: la suma de los valores de exps
    return {tok: round(e / total, 3) for tok, e in exps.items()}

puntajes = {"Montevideo": 4.0, "una": 2.0, "la": 1.5}   # «La capital de Uruguay es ...»
for T in (0.5, 1.0, 2.0):
    print(f"T={T}:", softmax(puntajes, T))
revisar("Ejercicio 7", softmax({"a": 1.0, "b": 1.0}), {"a": 0.5, "b": 0.5}, "total = sum(exps.values())")

**Pregunta para pensar (no se corrige):** con `T = 0.5`, ¿la respuesta «una» es
imposible o solo muy improbable? ¿Qué te dice eso sobre por qué un LLM a veces
dice algo raro?

Probá `softmax(puntajes, 0)`: da error, porque divide por cero. Por eso T = 0
no usa la fórmula: elige siempre la más probable (mirá la guía de la clase 1).

Esa tabla y ese softmax son, en miniatura, lo que hace un LLM: **predice el próximo token** y elige según las
probabilidades. Por eso puede escribir algo fluido y falso: elige lo **probable**, no lo **verdadero**.

## Parte 3 · Más de cerca: tokens, temperatura y JSON (15 min)

### Ejercicio 8 · Tokens no son palabras

El proveedor cobra por **token**, no por palabra. Con lo de la Parte 1
(`.split()` y `len`) contá las palabras de una respuesta más larga y comparalas
con sus tokens.

**Tu turno:** Guardá en `palabras_respuesta` cuántas palabras tiene `r2.text`.

**Qué deberías ver** (21/10): una explicación de dos oraciones, con
54 palabras y 62 tokens de salida. En español, más tokens que
palabras: cada pregunta cuesta un poco más que en inglés.

In [ ]:
r2 = modelo.invoke("Explicá en dos oraciones qué es un token en un modelo de lenguaje.")
print(r2.text)

palabras_respuesta = None  # ← tu respuesta
print("palabras:", palabras_respuesta, "· tokens de salida:", r2.usage_metadata["output_tokens"])
revisar("Ejercicio 8", palabras_respuesta, len(r2.text.split()), "len(r2.text.split())")

### Ejercicio 9 · La temperatura, de verdad

Es la `T` del ejercicio 7, ahora en un modelo real. Va con Gemini 3.1
Flash-Lite, porque el 3.5 (el de la celda de preparar) la ignora. La celda le pide 3 veces lo mismo con
`T = 0` y 3 veces con `T = 1`: son 6 llamadas.

**Tu turno:** Guardá en `distintas` cuántas respuestas **distintas** hay en `respuestas`
(pista: un `set` guarda cada elemento una sola vez).

**Qué deberías ver:** con `T = 0`, 1 distinta: el mismo nombre las 3 veces. Con
`T = 1`, 2 o 3 distintas. Medido con 5 pedidos (`… clase1 temperatura`): `T = 0` dio
«Montevideo Mío» las 5 veces, en tres corridas; `T = 1`, 2, 4 y 3 distintos. Tus
nombres pueden ser otros: lo que se repite es que a más temperatura, más variedad.

Si dice `503 UNAVAILABLE` («high demand»), el 3.1 está saturado (nos pasó el 21/10):
esperá un minuto y volvé a correr la celda. Si sigue, pasá al 10: la temperatura
la volvés a probar en el laboratorio.

In [ ]:
pedido = "Inventá un nombre para una cafetería en Montevideo. Respondé solo el nombre."
for T in (0.0, 1.0):
    modelo_t = init_chat_model("google_genai:gemini-3.1-flash-lite", temperature=T)
    respuestas = [modelo_t.invoke(pedido).text.strip() for _ in range(3)]
    distintas = None  # ← tu respuesta
    print(f"T={T}: {respuestas} → {distintas} distintas")
revisar("Ejercicio 9", distintas, len(set(respuestas)), "distintas = len(set(respuestas))")

### Ejercicio 10 · JSON: cómo viaja todo esto

Lo que acabás de hacer viaja por internet: tu pedido y la respuesta.
Los dos viajan como **JSON**, un texto que se parece mucho a un diccionario de
Python. `json.dumps` convierte diccionario → texto; `json.loads`, texto →
diccionario.

**Tu turno:** La variable `respuesta_texto` es una respuesta real simplificada. Sacá el
texto que escribió el modelo y cuántos tokens de salida usó.

In [ ]:
import json

pedido = {"modelo": "un-modelo", "mensajes": [{"rol": "user", "contenido": "¿Qué es un token?"}],
          "tope_tokens": 300}  # cuánto puede escribir, como max_salida en config.toml
print("Lo que se envía:", json.dumps(pedido, ensure_ascii=False))

respuesta_texto = '{"contenido": "Un pedazo de texto que el modelo procesa.", "uso": {"tokens_entrada": 12, "tokens_salida": 9}}'
respuesta = json.loads(respuesta_texto)

texto_del_modelo = None  # ← tu respuesta
tokens_salida = None     # ← tu respuesta
print(texto_del_modelo, tokens_salida)
revisar("Ejercicio 10", (texto_del_modelo, tokens_salida),
        ("Un pedazo de texto que el modelo procesa.", 9),
        'respuesta["contenido"] y respuesta["uso"]["tokens_salida"]')

## Tu diagnóstico (1 min)

Sale solo de los ejercicios: no hay que completar nada. Corré la celda y copiá lo que imprime en `evidencia/m1.md`
(práctica M1). Sirve para saber qué repasar: no tiene nota.

In [ ]:
bien = [nombre for nombre, ok in resultados.items() if ok]
TEMAS = {1: "variables", 2: "listas", 3: "for e if", 4: "diccionarios", 5: "funciones",
         6: "tabla de qué sigue", 7: "softmax", 8: "tokens", 9: "temperatura", 10: "JSON"}  # qué repasar en la guía
repasar = [f"Ejercicio {i} ({TEMAS[i]})" for i in range(1, 11) if not resultados.get(f"Ejercicio {i}")]
print("Te salió:", ", ".join(bien) or "todavía ninguno")
print("Para repasar:", ", ".join(repasar) or "nada")

## Qué sigue

En la segunda mitad de la clase instalás el proyecto en tu computadora y hacés
lo mismo desde ahí: `check_setup.py` le pregunta al mismo Gemini, con la misma
clave (la pegás en `.env`). Después, en el laboratorio, cambiás una cosa por vez
y mirás qué pasa.

Si te quedaron ejercicios sin terminar, terminalos en casa: la clase 2 los da
por sabidos. La guía de la clase 1 (en el aula virtual) explica todo de nuevo
con calma.